# 1. ChatPromptTemplate 的使用
## 1. ChatPromptTemplate 的实例化
### 1.1 from_messages()
## 2. ChatPromptTemplate 的调用
### 2.1 involke()
传入参数：字典列表；返回值类型：ChatPromptValue
### 2.2 format()
参数类型：变量值；返回值类型：字符串
### 2.3 format_messages()
参数类型：变量值；返回值类型：消息列表

In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek-flash",
    model_provider="openai",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

# 定义聊天提示词模版
chat_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个有帮助的AI机器人，你的名字是{name}。"),
    ("human", "你好，最近怎么样？"),
    ("ai", "我很好，谢谢！"),
    ("human", "{user_input}"),
])

# 格式化聊天提示词模版中的变量，返回值：ChatPromptValue
prompt1 = chat_template.invoke({"name":"小明", "user_input":"你叫什么名字？"})
# 格式化聊天提示词模版中的变量，返回值：str
prompt2 = chat_template.format(name="小明", user_input="你叫什么名字？")
# 格式化聊天提示词模版中的变量，返回值：消息列表（list[BaseMessage]）
prompt3 = chat_template.format_messages(name="小明", user_input="你叫什么名字？")
# 打印格式化后的聊天提示词模版内容
print(prompt1)
print(prompt2)
print(prompt3)

chat_template.invoke(prompt1)

messages=[SystemMessage(content='你是一个有帮助的AI机器人，你的名字是小明。', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，最近怎么样？', additional_kwargs={}, response_metadata={}), AIMessage(content='我很好，谢谢！', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你叫什么名字？', additional_kwargs={}, response_metadata={})]


### 2.4 初始化综合使用

In [ ]:
from langchain_core.prompts import (ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate,)
from langchain_core.messages import SystemMessage, HumanMessage

# 示例 1: 使用 BaseMessage（已实例化的消息），不可用占位符
system_msg = SystemMessage(content="你是一个AI工程师。")
human_msg = HumanMessage(content="你好！")

# 示例 2: 使用 BaseMessagePromptTemplate，可以使用占位符
system_prompt = SystemMessagePromptTemplate.from_template("你是一个{role}.")
human_prompt = HumanMessagePromptTemplate.from_template("{user_input}")

# 示例 3: 使用 BaseChatPromptTemplate（嵌套的 ChatPromptTemplate）
nested_prompt = ChatPromptTemplate.from_messages([("system", "嵌套提示词")])

prompt = ChatPromptTemplate.from_messages([
    system_msg,    # MessageLike (BaseMessage)
    human_msg,     # MessageLike (BaseMessage)
    system_prompt, # MessageLike (BaseMessagePromptTemplate)
    human_prompt,  # MessageLike (BaseMessagePromptTemplate)
    nested_prompt, # MessageLike (BaseChatPromptTemplate)
])
prompt.invoke({"role":"人工智能专家","user_input":"介绍一下大模型的应用场景"})

# 3. 高级特性
## 3.1 部分变量预填充：partial()
为不同场景的用户创建不同的模版

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 原始模板
template = ChatPromptTemplate.from_messages([
    ("system", "你是{role}，目标用户是{audience}"),
    ("user", "{task}")
])

# 部分填充
customer_support_template1 = template.partial(role="客服专员", audience="普通用户")
customer_support_template2 = template.partial(role="销售顾问", audience="意向用户")

# 现在只需要提供 task
messages1 = customer_support_template1.invoke({"task": "解释退款政策"})
messages2 = customer_support_template1.invoke({"task": "介绍下我发你的这款产品"})
messages3 = customer_support_template2.invoke({"task": "我想买一台凯迪拉克XT5，有什么优惠吗？"})

print(messages1)
print(messages2)
print(messages3)

## 3.2. 消息占位符——placeholder
JSON 形式

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "你是一个有用的AI助手"),
    ("placeholder", "{conversation}"),
])

prompt_value = template.invoke(
    {
        "conversation": [
            ("human", "你好!"),
            ("ai", "今天我能帮你做什么？"),
            ("human", "你能给我做一个冰激凌吗？"),
            ("ai", "抱歉，我没有这样的能力"),
        ]
    }
)

print(prompt_value)

MessagesPlaceholder 形式

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder("msgs")
])

prompt_template.invoke({"msgs": [HumanMessage(content="hi!")]})
# prompt_template.format_messages(msgs=[HumanMessage(content="hi!")])

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 存储对话历史内容
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个非常友好的AI助手"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}")
    ]
)
prompt_template.invoke(
    {
        "history": [
            ("human", "5 + 2 = ?"),
            ("ai", "5 + 2 = 7")
        ],
        "question": "结果再乘以4呢？"
    }
)

## 3.3. 可服用模版库
定义了具体的存放模版的 py 文件

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
class PromptLibrary:
    """可复用的提示词模板库"""

    TRANSLATOR = ChatPromptTemplate.from_messages([
        ("system", "你是专业翻译，精通{source_lang}和{target_lang}"),
        ("user", "翻译以下文本：\n{text}")
    ])

    CODE_REVIEWER = ChatPromptTemplate.from_messages([
        ("system", "你是{language}代码审查专家，重点关注{focus}"),
        ("user", "审查代码：\n```{language}\n{code}\n```")
    ])

    SUMMARIZER = ChatPromptTemplate.from_messages([
        ("system", "你是内容摘要专家"),
        ("user", "将以下内容总结为{num}个要点：\n{content}")
    ])

    TUTOR = ChatPromptTemplate.from_messages([
        ("system", "你是{subject}导师，学生水平：{level}"),
        ("user", "{question}")
    ])

其它文件中使用

In [ ]:
# from templates import PromptLibrary

messages = PromptLibrary.TRANSLATOR.format_messages(
    source_lang="英语",
    target_lang="中文",
    text="Hello World"
)

In [ ]:
# templates/
# ├── __init__.py
# ├── common.py # 通用模板
# ├── translation.py # 翻译相关
# └── coding.py # 编程相关

# common.py 文件中示例
from langchain_core.prompts import ChatPromptTemplate

FRIENDLY_ASSISTANT = ChatPromptTemplate.from_messages([
    ("system", "你是一个友好的助手"),
    ("user", "{input}")
])